# Baseline age model: train and evaluate (tutorial)

This notebook is for **competitors and teachers** learning how the repository’s baseline pipeline works.

**What you will do**
1. Check that prepared pseudobulk data exists.
2. Run `models/train_age_model.py` (Random Forest on donor-level pseudobulk).
3. Run `models/evaluate_test.py` to compare test predictions to labels (when available).
4. Inspect outputs (metrics, optional top genes).

**How to run**
- Open this notebook from the project root `aging_challenge_2026/`, or the first code cell sets paths relative to the repo.
- Execute cells **in order** (top to bottom).

**Time / resources**  
- Training uses CPU RAM and can take several minutes depending on `--n-genes` and `--n-estimators`. On a login node, use modest settings first (defaults are reasonable).

## 1. Concepts: what the baseline does

### Input data
- **File:** `data/pseudobulk/combined_pseudobulk_donor_aggregated.h5ad`
- **Rows:** one row per **donor** (not per cell).
- **Columns (features):** for each gene, there are **five** pseudobulk expression values—one per **cell-type group** (e.g. CD4 T, CD8 T, NK, B, Monocytes). Feature names look like `ENSG...__CellTypeGroup`.
- **`obs` columns:** includes `age`, `donor_id`, and **`_split`** with values `train`, `val`, or `test`.

### Training script (`train_age_model.py`)
- **Model:** scikit-learn `RandomForestRegressor` (regression → predicted age in years).
- **Feature selection (default):** keeps the top **N genes by variance** across donors (then keeps all five cell-type columns for those genes). This reduces dimensionality and runtime.
- **Alternative:** `--all-features` uses every gene (much heavier).
- **Preprocessing:** `log1p` on pseudobulk counts.
- **Splitting:** only `train` rows are used to fit the forest; `val` rows are used to print validation metrics; `test` rows are used to **predict** ages (saving `test_predictions.csv`). Test **labels** are not used during training.

### Evaluation script (`evaluate_test.py`)
- Merges predicted ages with **ground-truth** ages by `donor_id`.
- **Ground-truth file:** `data/test_labels_hidden.csv` — in the real challenge this may be **organiser-only**; locally you need this file (or your own truth CSV) to compute metrics.
- Reports **MAE**, **RMSE**, **R²**, **Pearson**, **Spearman** (see `models/README.md` for metric explanations).
- Writes `test_eval_metrics.csv` (and optionally a scatter plot) **next to** the predictions file you evaluated.

## 2. Environment: paths and Python

We locate the **repository root** so subprocess calls match `python models/train_age_model.py` from the project root.

**Teaching note:** In HPC environments, activate the same conda/env that has `scanpy`, `sklearn`, `scipy` installed before starting Jupyter.

In [1]:
import sys
import subprocess
from pathlib import Path

# If the notebook lives in notebooks/, parents[1] is the repo root
NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model.py. Open Jupyter from the repo root or from notebooks/."
    )

DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Python:", sys.executable)

Project root: /iridisfs/ddnb/Ahmed/aging_challenge_2026
Python: /usr/bin/python


## 3. Check prerequisites

Before training, the **donor-aggregated pseudobulk** AnnData must exist (build it with `data_prep/h5ad_to_pseudobulk.py` after `prepare_onek1k.py` if you have not already).

For **evaluation**, `test_labels_hidden.csv` must exist if you want metrics (competition organisers hold the official test labels).

In [2]:
H5AD_PATH = DATA_DIR / "pseudobulk" / "combined_pseudobulk_donor_aggregated.h5ad"
TRUTH_PATH = DATA_DIR / "test_labels_hidden.csv"

print("Pseudobulk h5ad exists:", H5AD_PATH.exists(), "→", H5AD_PATH)
print("Test labels exist:", TRUTH_PATH.exists(), "→", TRUTH_PATH)

if not H5AD_PATH.exists():
    print(
        "\n⚠ Train script will fail until this file exists. Copy/bind competition data into data/ (see data/README.md)."
    )

Pseudobulk h5ad exists: True → /iridisfs/ddnb/Ahmed/aging_challenge_2026/data_prep/output/pseudobulk/combined_pseudobulk_donor_aggregated.h5ad
Test labels exist: True → /iridisfs/ddnb/Ahmed/aging_challenge_2026/data_prep/output/test_labels_hidden.csv


## 4. Run training (`train_age_model.py`)

This cell runs the **same** entry point as the command line:

```bash
python models/train_age_model.py [options]
```

**Parameters you can change below**
- `N_GENES` — how many highest-variance genes to keep (ignored if `ALL_FEATURES=True`).
- `N_ESTIMATORS` — number of trees in the forest (more trees → often stabler but slower).
- `ALL_FEATURES` — set `True` to disable gene filtering (large memory).
- `OUTPUT_DIR` — set to `None` for a timestamped folder under `results/`; or a fixed path string for reproducible reruns.

**Teaching prompt:** Why do we validate on `val` but report final numbers on `test`? (Avoid tuning on the test set; test estimates generalisation.)

In [3]:
from datetime import datetime

# ---- Tune these for class demos / quick tests ----
N_GENES = 2000
N_ESTIMATORS = 100
ALL_FEATURES = False
SEED = 42
# Optional: set to a string path to pin outputs, e.g. str(RESULTS_DIR / "demo_run")
OUTPUT_DIR = None  # None → timestamped folder under results/

train_py = PROJ_ROOT / "models" / "train_age_model.py"
cmd = [sys.executable, str(train_py)]
cmd += ["--input", str(H5AD_PATH)]
cmd += ["--n-genes", str(N_GENES), "--n-estimators", str(N_ESTIMATORS), "--seed", str(SEED)]
if ALL_FEATURES:
    cmd.append("--all-features")
if OUTPUT_DIR:
    cmd += ["--output-dir", str(OUTPUT_DIR)]
else:
    run_dir = RESULTS_DIR / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    cmd += ["--output-dir", str(run_dir)]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
if result.returncode != 0:
    raise RuntimeError("train_age_model.py exited with code %s" % result.returncode)
print("\nTraining finished successfully.")

Running: /usr/bin/python /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/train_age_model.py --n-genes 2000 --n-estimators 100 --seed 42
Output directory: /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/2026-03-25_12-45-53
Loading /iridisfs/ddnb/Ahmed/aging_challenge_2026/data_prep/output/pseudobulk/combined_pseudobulk_donor_aggregated.h5ad...
  Shape: 981 donors × 141390 features (5 values per gene per cell type)
Selecting top 2000 genes by variance...
  Using 10000 features
Train: 781, Val: 95, Test: 105
Training Random Forest...

Validation metrics:
  MAE:     9.205
  RMSE:    11.992
  R²:      0.474
  Pearson: 0.696
  Spearman: 0.621

Test predictions saved to /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/2026-03-25_12-45-53/test_predictions.csv
Model saved to /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/2026-03-25_12-45-53/rf_age_model.joblib
Top 20 genes saved to /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/2026-03-25_12-45-53/top_genes.csv

## 5. Locate the latest run and preview predictions

By default, this notebook sends outputs to `results/YYYY-MM-DD_HH-MM-SS/` (via `--input` and `--output-dir`).

**Files written**
- `test_predictions.csv` — `donor_id`, `predicted_age` for the **test** split
- `rf_age_model.joblib` — fitted model
- `top_genes.csv` — top genes by Random Forest **feature importance** (aggregated across cell-type columns)

In [4]:
import pandas as pd
from IPython.display import display

out_base = RESULTS_DIR
legacy_pred = out_base / "test_predictions.csv"

if legacy_pred.exists():
    run_dir = out_base
    pred_path = legacy_pred
else:
    subdirs = sorted([d for d in out_base.iterdir() if d.is_dir()], reverse=True)
    run_dir = None
    pred_path = None
    for d in subdirs:
        p = d / "test_predictions.csv"
        if p.exists():
            run_dir = d
            pred_path = p
            break

if pred_path is None:
    raise FileNotFoundError("No test_predictions.csv found under results/")

print("Run directory:", run_dir)
print("Predictions file:", pred_path)
display(pd.read_csv(pred_path).head(10))

Run directory: /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output
Predictions file: /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/test_predictions.csv


,donor_id,predicted_age
0,10,71.93
1,106,40.02
2,111,68.66
3,120,57.01
4,125,64.18
5,127,40.09
6,131,53.16
7,148,53.38
8,152,41.31
9,153,59.07


## 6. Run evaluation (`evaluate_test.py`)

This merges predictions with `test_labels_hidden.csv` and prints regression metrics.

**Teaching notes**
- If `test_labels_hidden.csv` is missing, evaluation **cannot** compute test metrics (typical for a blind leaderboard).
- For `--plot`, matplotlib must be installed; the PNG is saved beside the predictions.
- You can pass an explicit `--predictions` path to score a specific run folder.

In [5]:
eval_py = PROJ_ROOT / "models" / "evaluate_test.py"

if not TRUTH_PATH.exists():
    print(
        "Skipping evaluation: test_labels_hidden.csv not found. "
        "Add ground-truth labels to run this step."
    )
else:
    cmd = [sys.executable, str(eval_py), "--predictions", str(pred_path)]
    # Uncomment to save scatter plot (requires matplotlib):
    # cmd.append("--plot")
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJ_ROOT))
    if result.returncode != 0:
        raise RuntimeError("evaluate_test.py exited with code %s" % result.returncode)

Running: /usr/bin/python /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/evaluate_test.py --predictions /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/test_predictions.csv
Test set evaluation (n=105 donors)

  MAE:      8.676
  RMSE:     10.695
  R²:       0.611
  Pearson:  0.825 (p=2.92e-27)
  Spearman: 0.749 (p=4.10e-20)

Metrics saved to /iridisfs/ddnb/Ahmed/aging_challenge_2026/models/output/test_eval_metrics.csv


## 7. Load saved metrics

Metrics are stored as a simple CSV next to the predictions file. **Interpret briefly:**
- **MAE** — average error in years.
- **R²** — fraction of age variance explained (higher is better, capped at 1 for perfect predictions).
- **Pearson / Spearman** — correlation between predicted and true age linearly vs. by rank.

In [6]:
from IPython.display import display

metrics_file = pred_path.parent / "test_eval_metrics.csv"
if metrics_file.exists():
    display(pd.read_csv(metrics_file))
else:
    print("No metrics file yet (run evaluation cell when truth labels exist).")

,metric,value
0,MAE,8.675524
1,RMSE,10.694509
2,R2,0.610673
3,Pearson,0.824847
4,Spearman,0.748877


## 8. Optional: top genes from the last training run

Feature importances are **not** causal; they indicate how much the forest used each gene’s pseudobulk signal in this pipeline. High overlap with biological ageing literature can be a sanity check.

In [7]:
from IPython.display import display

top_path = pred_path.parent / "top_genes.csv"
if top_path.exists():
    display(pd.read_csv(top_path))
else:
    print("No top_genes.csv beside this run.")

,rank,gene,importance
0,1,ENSG00000198535,0.116723
1,2,ENSG00000164327,0.055354
2,3,ENSG00000239779,0.040900
3,4,ENSG00000139890,0.033852
4,5,ENSG00000141034,0.018490
5,6,ENSG00000228716,0.018258
6,7,ENSG00000205927,0.017672
7,8,ENSG00000287358,0.015551
8,9,ENSG00000167332,0.009869
9,10,ENSG00000091317,0.008419


## Further ideas for competitors

- Swap the Random Forest for another regressor (e.g. gradient boosting, elastic net on PCA).
- Engineer features (pathways, cell-type proportions) instead of raw pseudobulk.
- Use cell-level models + pooling; compare to this donor-aggregated baseline.
- Always keep test labels **untouched** until final model choices are frozen.

See `models/README.md` and the main `README.md` for paths and data structure.